# Biomedical Image Analysis: **Vision-Language Prompting and Robustness**

**Objective**

In this notebook we attempt to ask a different question than standard image segmentation tasks:

> **How reliably can a vision-language model describe microscopy images when restricted strictly to observable features?**

In this task, we will be comparing an unstructured prompt to a structured prompt for consistency of information provided. We will also look at the validity of the JSON outputs, the repeatability of those results, and how the model deals with varying levels of image degradation.

The task is purely descriptive: the user is instructed not to engage in diagnostic or medical reasoning. Let's see how the model follows these instructions.

This is a **image description task**, not a diagnostic or clinical task, and responses should not be expected or requested to provide medical information or diagnosis of any kind.

### **1. Project Setup**

This notebook is designed to work with a previously prepared dataset from `task1_setup_eda.ipynb`. The following dataset checks are therefore reduced to a minimum and allow the notebook to run even if the required files are already stored in the Colab environment.

from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install -q ollama pandas matplotlib

from pathlib import Path
import sys
import os
import json
import base64
import subprocess
import time
import pandas as pd

USE_DRIVE = True  # Set to False to skip Drive entirely and upload manually each session

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/biomedical_image_project")
else:
    PROJECT_DIR = Path("/content/biomedical_image_project")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR = PROJECT_DIR / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)

if USE_DRIVE and (SRC_DIR / "biomedical_utils.py").exists():
    print("Found biomedical_utils.py on Drive, no upload needed.")
else:
    print("Please upload biomedical_utils.py")
    from google.colab import files
    uploaded = files.upload()
    import shutil
    shutil.move("biomedical_utils.py", str(SRC_DIR / "biomedical_utils.py"))

sys.path.insert(0, str(SRC_DIR))
from biomedical_utils import ensure_dataset, load_metadata, extract_and_validate_json

DATA_DIR = ensure_dataset()
metadata = load_metadata(DATA_DIR)

print("Dataset:", DATA_DIR)

Mounted at /content/drive
Found biomedical_utils.py on Drive, no upload needed.
Dataset: /content/nuclei_dataset


### **2. Start the local Ollama service**


In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.24.0 sh

server_running = subprocess.run(
    ["pgrep", "-f", "ollama serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0

if not server_running:
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(5)

!ollama pull llama3.2-vision
!ollama list

Selecting previously unselected package zstd.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

NAME                      ID              SIZE      MODIFIED               
llama3.2-vision:latest    6f2f9757ae97    7.8 GB    Less than a second ago    


In [ ]:
import ollama

health_check = ollama.chat(
    model="llama3.2-vision",
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
)
print("Model response:", health_check["message"]["content"])

Model response: Go!


### **3. Choose a representative image**

We decide to choose the first one of the first training image labelled `normal` as the representative image. By using an image from the training set we also make sure that this exploratory analysis is separate from our final evaluation on the test set.

In [ ]:
representative_row = (
    metadata.loc[
        (metadata["split"] == "train") & (metadata["density"] == "normal")
    ]
    .iloc[0]
)

representative_image = (
    DATA_DIR / "train" / "images" / f"{representative_row['image_id']}.png"
)

print("Representative image:", representative_image)
display(representative_row.to_frame().T)

Representative image: /content/nuclei_dataset/train/images/train_001.png


,image_id,split,density,n_objects,mean_intensity,area_fraction,seed
1,train_001,train,normal,29,0.73613,0.086044,20260716


In [ ]:
def encode_image(path):
    with open(path, "rb") as file:
        return base64.b64encode(file.read()).decode("utf-8")

image_b64 = encode_image(representative_image)

### **4. Begin with a naive prompt**

Let's assume that a simplest possible prompt would be something like

> **What do you see in this image?**

This would set the bar to the lowest possible, the model is not expected to answer the question in any particular detail or follow specific conventions. It would just describe what it "sees" in the image in a generic manner.

In [ ]:
naive_prompt = "What do you see in this image?"

naive_response = ollama.chat(
    model="llama3.2-vision",
    messages=[{
        "role": "user",
        "content": naive_prompt,
        "images": [image_b64],
    }],
)

print(naive_response["message"]["content"])

The image appears to show a microscopic view of cells, with blue dots representing the nucleus of each cell. The image is likely a fluorescence microscopy image, where the blue color indicates the presence of a specific protein or molecule within the cells. The image may be used to study the distribution and behavior of this protein or molecule within the cells.


### **5. Create a restricted prompt**

The improved prompt should aim to restrict the model’s responses in three ways:

1. Encourage the model to describe only what it actually sees in the image;
2. Discourage the model from making medical diagnoses or suggesting treatments;
3. Enforce a strict JSON format for the response.

The last point is important because it ensures the response can be validated and standardized.

In [ ]:
optimised_prompt = """
You are looking at a microscope image captured using fluorescence microscopy.
Cell nuclei are visible as bright spots on a dark background.

Describe ONLY those things that you see in the image. Do not give medical opinions
or interpretations. Do not attempt to identify any diseases or abnormalities
that may be indicated by the image. You are simply describing what you see.

If you are uncertain about any part of the image, state “uncertain” instead of speculating.

Provide only a valid JSON object in this exact format in your response; do not
include any extra text.:

{
  "modality": "the imaging modality used, or 'uncertain' if this cannot be determined",
  "tissue_type": "the biological structure visible, described observationally,
  or 'uncertain' if cannot be determined",
  "notable_features": "any notable features, such as density, diffuseness or
  pattern of any observed tissue, and contrast of different tissues.",
  "image_quality": "the technical quality of the image, such as sharpness, focus,
  contrast or lack of noise, or "uncertain" if cannot be determined."
}
""".strip()

optimised_response = ollama.chat(
    model="llama3.2-vision",
    messages=[{
        "role": "user",
        "content": optimised_prompt,
        "images": [image_b64],
    }],
)

print(optimised_response["message"]["content"])

{
  "modality": "fluorescence microscopy",
  "tissue_type": "cell nuclei",
  "notable_features": "bright spots on a dark background, uniform in size and shape",
  "image_quality": "good, with clear and sharp details, but with some noise visible"
}


In [ ]:
required_fields = ["modality", "tissue_type", "notable_features", "image_quality"]

### **6. Test Repeatability**

Knowing vision-language models can produce different descriptions for the same image, we have to determine if the model is behaving consistently.

Here, we attempt the same constrained prompt 5 times and analyze the results. We think this is more informative than simply showing a single successful prompt, as it demonstrates that the model behaves consistently across different runs.

In [ ]:
n_runs = 5
repeatability_results = []

for run_number in range(1, n_runs + 1):
    response = ollama.chat(
        model="llama3.2-vision",
        messages=[{
            "role": "user",
            "content": optimised_prompt,
            "images": [image_b64],
        }],
    )

    text = response["message"]["content"]
    parsed, status = extract_and_validate_json(text, required_fields)

    repeatability_results.append({
        "run": run_number,
        "valid_json": parsed is not None,
        "parsed": parsed,
        "status": status,
    })

    print(f"Run {run_number}: {status}")

print("\nField-by-field comparison:")
for field in required_fields:
    print(f"\n{field}:")
    for result in repeatability_results:
        value = (
            result["parsed"].get(field, "missing")
            if result["parsed"]
            else "PARSE FAILED"
        )
        print(f"  Run {result['run']}: {value}")

Run 1: Valid, all required fields present
Run 2: Valid, all required fields present
Run 3: Valid, all required fields present
Run 4: Valid, all required fields present
Run 5: Valid, all required fields present

Field-by-field comparison:

modality:
  Run 1: Fluorescence microscopy
  Run 2: fluorescence microscopy
  Run 3: Fluorescence microscopy
  Run 4: fluorescence microscopy
  Run 5: fluorescence microscopy

tissue_type:
  Run 1: Cell nuclei
  Run 2: cell nuclei
  Run 3: Cell nuclei
  Run 4: cell nuclei
  Run 5: cell nuclei

notable_features:
  Run 1: Bright, circular, and uniformly distributed
  Run 2: bright spots on a dark background
  Run 3: Bright spots on a dark background, uniform size and shape
  Run 4: bright spots on a dark background
  Run 5: bright spots on a dark background, uniform size and shape

image_quality:
  Run 1: Good, with clear and sharp details
  Run 2: good, with clear and sharp details
  Run 3: Good, with clear contrast and sharpness
  Run 4: good, with cl

**Interpretation**

All five runs again produced correct JSON with all required fields, and there were no errors this time.

The modality and tissue were the same (“fluorescence microscopy”) and (“cell nuclei”) for all the runs.
The image quality was good for four runs, while the fifth run reported it as uncertain.
The notable features were mostly the same, though they were worded differently in different runs.
This shows that the prompt was effective in prompting the model to consistently report the important information.

However, there were variations in the descriptions and confidence levels for the minor features.

### **8. Test image quality under degradation**

The dataset contains several versions of the selected test images that are blurred or have the low-contrast.

These versions allow us to test a more interesting aspect of robustness

> **If the visual quality of the input image is reduced, will the model correctly reflect this in its result?**

In [ ]:
corrupted_dir = DATA_DIR / "test_corrupted" / "images"
corrupted_files = sorted(os.listdir(corrupted_dir))

print("Corrupted images:")
print(corrupted_files)

test_images = {
    "clean": DATA_DIR / "test" / "images" / "test_000.png",
    "low_contrast": corrupted_dir / "test_000_lowcontrast.png",
    "blurred": corrupted_dir / "test_000_blur.png",
}

comparison_results = {}

for label, path in test_images.items():
    response = ollama.chat(
        model="llama3.2-vision",
        messages=[{
            "role": "user",
            "content": optimised_prompt,
            "images": [encode_image(path)],
        }],
    )

    text = response["message"]["content"]
    parsed, status = extract_and_validate_json(text, required_fields)

    comparison_results[label] = {
        "valid_json": parsed is not None,
        "parsed": parsed,
        "status": status,
        "uncertain_used": "uncertain" in text.lower(),
    }

for label, result in comparison_results.items():
    print(f"\n{label}:")
    print("  Valid JSON:", result["valid_json"])
    print("  Explicit uncertainty used:", result["uncertain_used"])
    print(
        json.dumps(result["parsed"], indent=2)
        if result["parsed"]
        else result["status"]
    )

Corrupted images:
['test_000_blur.png', 'test_000_lowcontrast.png', 'test_004_blur.png', 'test_004_lowcontrast.png']

clean:
  Valid JSON: True
  Explicit uncertainty used: False
{
  "modality": "fluorescence microscopy",
  "tissue_type": "cell nuclei",
  "notable_features": "bright spots on a dark background",
  "image_quality": "good, with clear and distinct cell nuclei"
}

low_contrast:
  Valid JSON: True
  Explicit uncertainty used: True
{
  "modality": "uncertain",
  "tissue_type": "uncertain",
  "notable_features": "bright spots of varying sizes and intensity",
  "image_quality": "uncertain"
}

blurred:
  Valid JSON: True
  Explicit uncertainty used: True
{
  "modality": "fluorescence microscopy",
  "tissue_type": "cell nuclei",
  "notable_features": "bright spots on a dark background",
  "image_quality": "uncertain"
}


In [ ]:
vlm_output = {
    "naive_response": naive_response["message"]["content"],
    "optimised_response": optimised_response["message"]["content"],
    "repeatability_results": repeatability_results,
    "corruption_comparison": comparison_results,
}

with open(PROJECT_DIR / "vlm_results.json", "w") as f:
    json.dump(vlm_output, f, indent=2, default=str)

print("Saved to:", PROJECT_DIR / "vlm_results.json")

Saved to: /content/drive/MyDrive/biomedical_image_project/vlm_results.json


**Interpretation**

Overall, the results demonstrate mixed outcomes. On the one hand, the model performed well on low-contrast images by raising an alert, as both the modality and image_quality variables were identified as uncertain. It shows that the model can recognize problems with contrast. On the other hand, the model failed to identify blurry images as low-quality. When presented with a blurry image, the model remained confident about its assessment, providing clear answers for the modality and tissue_type variables and deeming the image_quality as good, despite the evident blurriness of the input.

This outcome highlights the model’s inability to distinguish blurry images from those of a good quality, a crucial distinction if one wishes to use the model for automatic screening of low-quality inputs.

### **9. What we can learn from this experiment**

The main lesson from this experiment is that while the constrained prompt is useful in enforcing the format of the response, the content of the response can be varied on different runs, even for the same image. Moreover, it is not always clear whether the model is confident about the results. The text descriptions of the blurred images were not significantly different from those of the clear images, which means that the model is not always aware of its own limitations

In biomedical imaging, this is especially important, because a seemingly correct response might be based on an incorrect image analysis.

### What is next?

We will leave the realm of unconstrained prompts in `task2_classical_segmentation.ipynb` and start building a baseline image analysis pipeline based on Otsu thresholding, morphological operations, connected component analysis, and region measurements. The quantitative data obtained from this pipeline can then be used to train the LLM to generate more reliable text descriptions.